# Lição 10 - Agentes de IA em Produção

Nesta lição irá aprender **padrões de produção** para agentes de IA usando o Microsoft Agent Framework (Python). Cobrimos:

- **Observabilidade** — adicionar medições de tempo e registo às interações do agente
- **Avaliação** — usar um agente avaliador para pontuar a qualidade das respostas
- **Gestão de custos** — estratégias para otimização de tokens e seleção de modelos

O cenário é um **agente de viagens** que ajuda os utilizadores a planear viagens, com monitorização e avaliação em camadas.


## Configuração


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Considerações de Produção

Levar agentes de IA de protótipos para produção requer atenção cuidadosa a três pilares:

1. **Observabilidade** — É necessário ter visibilidade sobre o que o agente está a fazer, quanto tempo demora, e que ferramentas chama. Sem rastreamento e registo, depurar problemas em produção é quase impossível.

2. **Avaliação** — Verificações automáticas de qualidade garantem que as respostas do agente se mantenham precisas, completas e úteis ao longo do tempo. Um agente avaliador pode pontuar as respostas com base em critérios definidos.

3. **Gestão de Custos** — A utilização de tokens impacta diretamente o custo. Estratégias como otimização de prompts, seleção do modelo e armazenamento em cache ajudam a manter as despesas sob controlo sem sacrificar a qualidade.


## Construir um Agente Observável

Definimos ferramentas de viagem e envolvemos a chamada ao agente com medição de tempo para que possamos monitorizar a latência. Em produção, integraria o OpenTelemetry ou um backend de rastreio semelhante.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Padrões de Avaliação

Um padrão comum em produção é usar um segundo agente como um **avaliador**. O avaliador classifica a resposta do agente principal com base em critérios pré-definidos como completude, precisão e utilidade.

Isto permite:
- Controlos de qualidade automatizados antes das respostas chegarem aos utilizadores
- Detecção de regressões quando os prompts ou modelos mudam
- Monitorização contínua do desempenho do agente ao longo do tempo


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Estratégias de Gestão de Custos

Controlar os custos é fundamental para agentes de IA em produção. Aqui estão as principais estratégias:

| Strategy | Description |
|---|---|
| **Otimização de prompts** | Mantenha as instruções do sistema concisas. Remova contexto redundante para reduzir os tokens de entrada. |
| **Seleção de modelo** | Utilize modelos mais pequenos e baratos (por exemplo, GPT-4o-mini) para tarefas simples como classificação ou extração, e reserve modelos maiores para raciocínio complexo. |
| **Armazenamento em cache** | Armazene em cache os resultados de ferramentas e consultas frequentes para evitar chamadas de API redundantes. |
| **Orçamentos de tokens** | Defina limites de `max_tokens` para evitar respostas inesperadamente longas. |
| **Agrupamento** | Agrupe várias consultas de utilizador numa única chamada de API sempre que possível. |

Na prática, uma abordagem por níveis funciona bem: encaminhe pedidos simples para um modelo rápido e económico e escale apenas as consultas complexas para um modelo mais capaz.


## Resumo

Nesta lição aprendeu a:

1. **Adicionar observabilidade** às interações dos agentes com temporização e registo, estabelecendo as bases para rastreio e monitorização.
2. **Avaliar as respostas do agente** automaticamente usando um agente avaliador que pontua completude, precisão e utilidade.
3. **Gerir custos** através da otimização de prompts, seleção de modelos, cache e orçamentos de tokens.

Estes padrões de produção ajudam a garantir que os seus agentes de IA são fiáveis, mensuráveis e económicos em grande escala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Isenção de responsabilidade**:
Este documento foi traduzido utilizando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos por garantir a precisão, tenha em atenção que traduções automatizadas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se uma tradução profissional efetuada por um tradutor humano. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações erradas decorrentes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
